### Print Swagger endpoint

In [3]:
import json
import os
import urllib.error
import urllib.request
from urllib.parse import urlsplit, urlunsplit

from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from dotenv import load_dotenv

load_dotenv()


def get_credential():
    try:
        credential = DefaultAzureCredential()
        credential.get_token("https://management.azure.com/.default")
        return credential
    except Exception:
        return InteractiveBrowserCredential()


def get_ml_client():
    subscription_id = os.getenv("SUBSCRIPTION")
    resource_group = os.getenv("RESOURCE_GROUP")
    workspace_name = os.getenv("WS_NAME")

    if not all([subscription_id, resource_group, workspace_name]):
        raise RuntimeError(
            "SUBSCRIPTION, RESOURCE_GROUP, and WS_NAME must be set in the environment."
        )

    return MLClient(
        get_credential(),
        subscription_id,
        resource_group,
        workspace_name,
    )


def build_swagger_url(scoring_uri):
    parsed = urlsplit(scoring_uri)
    return urlunsplit((parsed.scheme, parsed.netloc, "/swagger.json", "", ""))


def fetch_swagger(swagger_url, api_key):
    headers = {
        "Accept": "application/json",
        "Authorization": f"Bearer {api_key}",
    }
    request = urllib.request.Request(swagger_url, headers=headers, method="GET")

    with urllib.request.urlopen(request) as response:
        return json.loads(response.read().decode("utf-8"))


endpoint_name = os.getenv(
    "ENDPOINT_NAME",
    "sklearn-diabetes-05141720427035",
)
ml_client = get_ml_client()
endpoint = ml_client.online_endpoints.get(endpoint_name)
api_key = (
    os.getenv("API_KEY")
    or ml_client.online_endpoints.get_keys(endpoint_name).primary_key
)
swagger_url = build_swagger_url(endpoint.scoring_uri)

print(f"Endpoint: {endpoint.name}")
print(f"Scoring URI: {endpoint.scoring_uri}")
print(f"Swagger URL: {swagger_url}")

try:
    swagger = fetch_swagger(swagger_url, api_key)
    print(json.dumps(swagger, indent=2))
except urllib.error.HTTPError as error:
    print(f"The request failed with status code: {error.code}")
    print(error.info())
    print(error.read().decode("utf-8", "ignore"))


GenAI tracing is not enabled. Set environment variable AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING=true to enable this experimental feature.
Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Endpoint: sklearn-diabetes-05141720427035
Scoring URI: https://sklearn-diabetes-05141720427035.eastus2.inference.ml.azure.com/score
Swagger URL: https://sklearn-diabetes-05141720427035.eastus2.inference.ml.azure.com/swagger.json
{
  "swagger": "2.0",
  "info": {
    "title": "ML service",
    "description": "API specification for the Azure Machine Learning service ML service",
    "version": "1.0"
  },
  "schemes": [
    "https"
  ],
  "consumes": [
    "application/json"
  ],
  "produces": [
    "application/json"
  ],
  "securityDefinitions": {
    "Bearer": {
      "type": "apiKey",
      "name": "Authorization",
      "in": "header",
      "description": "For example: Bearer abc123"
    }
  },
  "paths": {
    "/": {
      "get": {
        "operationId": "ServiceHealthCheck",
        "description": "Simple health check endpoint to ensure the service is up at any given point.",
        "responses": {
          "200": {
            "description": "If service is up and running, this r

### Save swagger json

In [9]:
import json
import os
import urllib.request

from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("API_KEY")
swagger_url = "https://sklearn-diabetes-05141720427035.eastus2.inference.ml.azure.com/swagger.json"
output_path = os.path.join(os.getcwd(), "swagger.json")


def fetch_swagger(swagger_url, api_key):
    headers = {
        "Accept": "application/json",
        "Authorization": f"Bearer {api_key}",
    }
    request = urllib.request.Request(swagger_url, headers=headers, method="GET")

    with urllib.request.urlopen(request) as response:
        return json.loads(response.read().decode("utf-8"))


swagger = fetch_swagger(swagger_url, api_key)
with open(output_path, "w", encoding="utf-8") as file:
    json.dump(swagger, file, indent=2)

print(f"Saved Swagger JSON to: {output_path}")
swagger



{'swagger': '2.0',
 'info': {'title': 'ML service',
  'description': 'API specification for the Azure Machine Learning service ML service',
  'version': '1.0'},
 'schemes': ['https'],
 'consumes': ['application/json'],
 'produces': ['application/json'],
 'securityDefinitions': {'Bearer': {'type': 'apiKey',
   'name': 'Authorization',
   'in': 'header',
   'description': 'For example: Bearer abc123'}},
 'paths': {'/': {'get': {'operationId': 'ServiceHealthCheck',
    'description': 'Simple health check endpoint to ensure the service is up at any given point.',
    'responses': {'200': {'description': "If service is up and running, this response will be returned with the content 'Healthy'",
      'schema': {'type': 'string'},
      'examples': {'application/json': 'Healthy'}},
     'default': {'description': 'The service failed to execute due to an error.',
      'schema': {'$ref': '#/definitions/ErrorResponse'}}}}},
  '/score': {'post': {'operationId': 'RunMLService',
    'description':

### Save  Swagger JSON file

In [10]:
import json
import os
import urllib.request

from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("API_KEY")
swagger_url = "https://sklearn-diabetes-05141720427035.eastus2.inference.ml.azure.com/swagger.json"
output_path = os.path.join(os.getcwd(), "swagger.json")


def fetch_swagger(swagger_url, api_key):
    headers = {
        "Accept": "application/json",
        "Authorization": f"Bearer {api_key}",
    }
    request = urllib.request.Request(swagger_url, headers=headers, method="GET")

    with urllib.request.urlopen(request) as response:
        return json.loads(response.read().decode("utf-8"))


swagger = fetch_swagger(swagger_url, api_key)
with open(output_path, "w", encoding="utf-8") as file:
    json.dump(swagger, file, indent=2)

print(f"Saved Swagger JSON to: {output_path}")
swagger

Saved Swagger JSON to: c:\develop\open-source\AILabs\az-ml\swagger.json


{'swagger': '2.0',
 'info': {'title': 'ML service',
  'description': 'API specification for the Azure Machine Learning service ML service',
  'version': '1.0'},
 'schemes': ['https'],
 'consumes': ['application/json'],
 'produces': ['application/json'],
 'securityDefinitions': {'Bearer': {'type': 'apiKey',
   'name': 'Authorization',
   'in': 'header',
   'description': 'For example: Bearer abc123'}},
 'paths': {'/': {'get': {'operationId': 'ServiceHealthCheck',
    'description': 'Simple health check endpoint to ensure the service is up at any given point.',
    'responses': {'200': {'description': "If service is up and running, this response will be returned with the content 'Healthy'",
      'schema': {'type': 'string'},
      'examples': {'application/json': 'Healthy'}},
     'default': {'description': 'The service failed to execute due to an error.',
      'schema': {'$ref': '#/definitions/ErrorResponse'}}}}},
  '/score': {'post': {'operationId': 'RunMLService',
    'description':

### Query Scoring endpoint

In [8]:
import json
import os
import urllib.error
import urllib.request

from dotenv import load_dotenv

load_dotenv()

# Request data goes here.
# Update the payload to match the schema expected by your endpoint.
data = {
    "data": [
        [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
        [10, 9, 8, 7, 6, 5, 4, 3, 2, 1],
    ]
}
body = json.dumps(data).encode("utf-8")

url = "https://sklearn-diabetes-05141720427035.eastus2.inference.ml.azure.com/score"
api_key = os.getenv("API_KEY")
if not api_key:
    raise RuntimeError("A key should be provided to invoke the endpoint.")

headers = {
    "Content-Type": "application/json",
    "Accept": "application/json",
    "Authorization": f"Bearer {api_key}",
}
request = urllib.request.Request(url, data=body, headers=headers, method="POST")

try:
    with urllib.request.urlopen(request) as response:
        print(response.read().decode("utf-8"))
except urllib.error.HTTPError as error:
    print(f"The request failed with status code: {error.code}")
    print(error.info())
    print(error.read().decode("utf-8", "ignore"))


[11055.977245525679, 4503.079536107787]


### Query Health Check endpoint

In [ ]:
import json
import os
import urllib.error
import urllib.request

from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("API_KEY")
url = "https://sklearn-diabetes-05141720427035.eastus2.inference.ml.azure.com/"


def fetch_health_check(url, api_key=None):
    headers = {"Accept": "application/json"}
    if api_key:
        headers["Authorization"] = f"Bearer {api_key}"

    request = urllib.request.Request(url, headers=headers, method="GET")

    with urllib.request.urlopen(request) as response:
        body = response.read().decode("utf-8")
        content_type = response.headers.get_content_type()

        if content_type == "application/json":
            return json.loads(body)

        return body


try:
    health_check = fetch_health_check(url, api_key)
    print(health_check)
except urllib.error.HTTPError as error:
    print(f"The request failed with status code: {error.code}")
    print(error.info())
    print(error.read().decode("utf-8", "ignore"))



The request failed with status code: 403
azureml-model-deployment: blue
azureml-model-session: blue
content-type: text/plain
content-length: 225
date: Mon, 25 May 2026 05:26:10 GMT
server: azureml-frontdoor
x-request-id: 753a60fc-1a56-4ee5-a807-4a8d11f91381
connection: close


key_auth_bad_header_forbidden
Please check this guide to understand why this error code might have been returned 
https://docs.microsoft.com/en-us/azure/machine-learning/how-to-troubleshoot-online-endpoints#http-status-codes

